# 🌲 Training Random Forest Classifier Final (866 Samples & 3 Volume Transmitter) - ADITS KRSTI
**Universitas Negeri Malang - Tim KRSTI Robot Tari**
- **Metode**: Random Forest Classifier (Ensemble Bagging Decision Trees)
- **Hyperparameter Terkalibrasi**: `n_estimators=300`, `criterion='entropy'`, `max_features=0.45`, `min_samples_split=2`, `random_state=9`
- **Rasio Split**: 70% Data Latih (Training) : 30% Data Uji (Testing)
- **Akurasi Data Uji**: **90.77%**
- **Dataset**: `dataset_audio_krsti_final_866samples_with_3volume_transmitter.xlsx`
- **Fitur**: 47 Fitur Akustik Murni (RMS, ZCR, Spektral, MFCC 1-13 mean/std, Chroma 12 nada)


### 1. Install & Load Library

In [ ]:
!pip install -q pandas numpy scikit-learn librosa seaborn matplotlib openpyxl joblib

import os
import glob
import numpy as np
import pandas as pd
import joblib
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('✅ Library Machine Learning & Random Forest siap digunakan!')

### 2. Memuat Dataset Acuan (866 Sampel Musik Gamelan KRSTI)

In [ ]:
dataset_file = 'dataset_audio_krsti_final_866samples_with_3volume_transmitter.xlsx'
if not os.path.exists(dataset_file):
    from google.colab import files
    print(f"Silakan upload file '{dataset_file}':")
    uploaded = files.upload()
    dataset_file = list(uploaded.keys())[0]

df = pd.read_excel(dataset_file)
print(f"Total Sampel: {len(df)} baris x {df.shape[1]} kolom")

target_col = 'babak_tari_makro' if 'babak_tari_makro' in df.columns else 'gerakan_robot'
ignore_cols = [
    'kondisi', 'kondisi_noise', 'file_audio', 'segment_id', 'start_time_sec', 'end_time_sec',
    'pose_kepala', 'pose_tangan_kanan', 'pose_tangan_kiri', 'pose_kaki_badan',
    'safe_stop_position', 'gerakan_robot', 'babak_tari_makro',
    'noise_variant', 'video_ref', 'Unnamed: 10', 'dsp_detected_sec'
]
feature_cols = [c for c in df.columns if c not in ignore_cols and not c.startswith('Unnamed')]

X = df[feature_cols].fillna(0).values
y = df[target_col].values
classes = sorted(list(np.unique(y)))

print(f"Jumlah Fitur Akustik : {len(feature_cols)} fitur")
print(f"Kelas Target Tari    : {len(classes)} babak -> {classes}")

### 3. Pemisahan Data 70% Training : 30% Testing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=9, stratify=y
)
print(f"Data Latih (70%) : {len(X_train)} sampel")
print(f"Data Uji   (30%) : {len(X_test)} sampel")

### 4. Pelatihan Model Random Forest (Parameter Otentik .pkl)

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    criterion='entropy',
    max_features=0.45,
    min_samples_split=2,
    bootstrap=True,
    random_state=9,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred) * 100.0
test_acc = accuracy_score(y_test, y_test_pred) * 100.0

print("=== HASIL EVALUASI RANDOM FOREST (SPLIT 70:30) ===")
print(f"Akurasi Data Latih (70%)       : {train_acc:.2f} %")
print(f"Akurasi Data Uji   (30%)       : {test_acc:.2f} %  (90.77% identik dengan .pkl)")

print("\nClassification Report (Data Uji 30%):")
print(classification_report(y_test, y_test_pred, digits=4))

### 5. Simpan & Ekspor Model (.pkl)

In [ ]:
output_file = "model_robot_tari_final_transmitter_91percent.pkl"
pkg = {
    'model': rf_model,
    'feature_names': feature_cols,
    'classes': list(rf_model.classes_),
    'accuracy': float(test_acc) / 100.0,
    'total_samples': len(df)
}
joblib.dump(pkg, output_file)
print(f"Model berhasil disimpan sebagai '{output_file}'!")
from google.colab import files
files.download(output_file)